WARNING!!!

The following code exploits a free Mosek licence the course "061652 ROBUST OPTIMIZATION" offered at Politecnico di Milano (expiration April 25th 2024). If you have error messages informing you about licencing issues, you may try uncommenting the installation lines for Gurobi. Otherwise, we recommend that you obtain your own licence of either Mosek (https://www.gurobi.com/) or Gurobi (https://www.mosek.com/products/trial/).

### Preliminaries

In [ ]:
# RUN THIS CELL ONLY ONCE TO SET UP THE ENVIRONMENT (FIRST TIME YOU OPEN THE NOTEBOOK)
# ------------------------------------------------------------------------------------- #
# INSTALL THE MAIN PACKAGES NEEDED FOR THE PROGRAM TO RUN i.e. RSOME (AND MOSEK OR GUROBI SOLVERS)
# Use "!pip" when using Google Colab
# Use "%pip" when running this notebook locally (better to avoid import errors if different Python environments are present locally)
# %pip install rsome
# !pip install mosek
# !rm mosek.lic
# !git clone https://github.com/erickdelage/80624
# !cp ./80624/mosek.lic .
# !rm -r ./80624
# !mkdir -p /root/mosek
# !cp ./mosek.lic /root/mosek
# Uncomment the following line to install Gurobi instead of MOSEK
# NB https://pypi.gurobi.com is deprecated, use the Gurobi website to create an account and get the installation instructions
#!pip install -i https://pypi.gurobi.com gurobipy

In [ ]:
# RUN THIS CELL ONLY ONCE TO SET UP THE ENVIRONMENT (FIRST TIME YOU OPEN THE NOTEBOOK)
# ------------------------------------------------------------------------------------- #
# Alternative: more robust installation method to install directly from Jupyter notebooks
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "rsome"])

In [1]:
# IMPORT THE MAIN PACKAGES NEEDED FOR THE PROGRAM TO RUN
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
import rsome as rso
from rsome import ro
import matplotlib.pyplot as plt
# from rsome import msk_solver as my_solver  #Import Mosek solver interface
# from rsome import grb_solver as my_solver  #Import Gurobi solver interface
from my_utilities import *  #Import all custom utility functions
from bmp_model import *  #Import BMP model functions


### Load and prepare the input data


In [2]:
# LOAD INPUT DATA FROM EXCEL FILE
# Declare the current date for the naming of the output files purposes
current_date = datetime.now().date() # Get the current date
# formatted_date = current_date.strftime('%d.%m.%Y') # Format the date as a string

# Load data from the Excel file using the custom utility function
file_path = 'C:/Users/lenovo/OneDrive - Politecnico di Milano/Work_cloud/DOTTORATO/Robust Optimization/Project/Project_database.xlsx'
data = read_excel_file(file_path)

In [3]:
# DEFINE SOME PROBLEM CONSTANTS AND THE BOUNDS FOR THE OPTMIMIZATION PROBLEM
n = data['Item characteristics'].shape[0] # number of items (co-feedstocks)
m = data['Ct'].shape[0] # number of available suppliers 

Vr = 2*1350+3179 # size of the biomethaneplant i.e. volume (m^3)
N = 365 # purchase horizon (days). Set this coherently with the data of the 'Availability' sheet in the Excel file!!
p_ch4_mwh = 110 # (fixed) biomethane selling price (€/MWh)
p_ch4 = p_ch4_mwh *0.01035 # (fixed) biomethane selling price converted in (€/Nm^{3}_{CH4})

bnds = np.array([
        [0, 120], # Constraints on total input diet volatile solids (VS (g_{VS}/ton_{FM}))
        [0.2, 4], # Constraints on total Kjeldahl nitrogen (TKN (kg_{N}/ton_{FM}))
        [20, 40], # Constraints on the carbon over nitrogen ratio (C/N (mol_C/mol_N))
        [0, 4], # Constraints on total ammonia nitrogen (TAN (kg_{N}/ton_{FM}))
        [5, 15], # Constraints on total alkalinity (TAC (kg_{CaCO3}/ton_{FM}))
        [0, 10], # Constraints on lipid content (LI (kg_{LI}/ton_{FM}))
        [0, 10], # Constraints on total volatile fatty acids (TVFA (kg_{Hac}/ton_{FM})...unit in acetate equivalent)
        [20,50] # Constraints on hydraulic retention time (HRT (days))
       ])

In [ ]:
# EXTRACT DATA FROM READING AND COMPUTE SOME QUANTITIES
VS = data['Item characteristics']['VS'].values.astype('float64') # Volatile Solids content (g_{VS}/ton_{FM})
BMPinf = data['Item characteristics']['BMPinf'].values.astype('float64') # Biochemical Methane Potential (BMP (Nm^3_{CH4}/ton_{FM}))
k = (np.concatenate(([data['Item characteristics'].values[:,3]], data['Item characteristics'].values[:,6:12].T), axis=0).T).astype('float64') # other substrate characteristics needed to enforce technical constraints
Cp = data['Purchase costs'].values[5:5+m,1:1+n].astype('float64') # feedstock purchase costs (€/ton_{FM})
A = data['Availability'].values[5:5+m,1:1+n].astype('float64') # feedstock availability matrix for each supplier (ton_{FM} over the entire purchase horizon N)
itemtype = data['Item characteristics'].values[:,-1] # Legislative cluster (P: product, S: slurry, P2: 2nd-turn cultures, W: waste, B: by-product)

# COMPUTE THE MATRIX OF TRANSPORTATION COSTS Ct
# Sum fixed tariff to 'distance tariff' (1 eur/km) and divide by 20 tons (nominal size of transportation truck)
Fixed_tariff = data['Item characteristics']['Fixed_tariff'].values.astype('float64') # fixed tariff (€/ton_{FM} for each type of feedstock)
d = data['Ct']['d'].values.astype('float64') # distance (km) from each supplier to the plant
d = d.reshape(-1, 1)
matrix = np.tile(Fixed_tariff, (m, 1))
Ct = (matrix + d)/20 # note: '20' is the nominal size of the transportation truck (tons) # rows: suppliers, columns: feedstocks

In [5]:
# CREATE THE BMP TIME CURVES FOR ALL THE FEEDSTOCKS
# RE-DECLARE SOME PARAMETERS TO CREATE THE BMP TIME CURVES
k_hydr = data['Item characteristics'].values[:,13].astype('float64') # hydrolysis rate constants (fast part of the BMP test i.e. of the rapidly biodegradable fraction) (1/day)
k_hyds = data['Item characteristics'].values[:,14].astype('float64') # hydrolysis rate constants (slow part of the BMP test i.e. of the slowly biodegradable fraction) (1/day)
BMPinf_r = data['Item characteristics'].values[:,15].astype('float64') # cumulative BMP produced by the rapidly biodegradable fraction (Nm^3_{CH4}/ton_{FM})
BMPinf_s = data['Item characteristics'].values[:,16].astype('float64') # cumulative BMP produced by the slowly biodegradable fraction (Nm^3_{CH4}/ton_{FM})

bmp_curve = [] # list to store the BMP time curves (1 point for each day of the time horizon)
for x0, y0, k1, k2 in zip(BMPinf_r, BMPinf_s, k_hydr, k_hyds): # loop over all the feedstocks
    t, bmp_day = evaluate_model(x0, y0, k1, k2, 50, 50) # evaluate the BMP model (two-pool first order kinetics model) over 50 days with 50 time points
    bmp_curve.append(bmp_day)

    # Plot the time response
    # fig, ax = plt.subplots(figsize=(12, 6))
    # ax.plot(t, bmp_day, label='BMP(t)', marker='o')
    # plt.xlabel('Time')
    # plt.ylabel('BMP(t)')
    # plt.legend()
    # plt.grid(True)
    # plt.show()
bmp_curve = np.array(bmp_curve, dtype=np.float64) # convert the list to a NumPy matrix for easier manipulation (rows: feedstocks, columns: time points)
#print(bmp_curve)

### Deterministic problem

"First simplified approach" (BMP = BMP$_{inf}$, no HRT effect)

In [ ]:
#Solve the nominal problem

# Find indices correspondent to a certain cluster to apply 'regulating authority constraints'
table1A = np.isin(itemtype,['S','B'])
table1B = np.isin(itemtype,['P'])
slurries = np.isin(itemtype,['S'])
secondharvest = np.isin(itemtype,['P2'])
Nminus1 = 1/N

#Create model
model=ro.Model('Biomethane supply-chain simplified')

#Define variables
x = model.dvar((n,m))          #

#List the objective and constraints
model.max(Nminus1*(p_ch4*(VS*BMPinf/1000)@x.sum(axis=1)-(Cp.T*x).sum()-(Ct.T*x).sum()))
model.st(x <= A.T)       # Availability constraint
for i in range(np.size(k,1)):
    model.st(k[:,i]@(x.sum(axis=1)) <= bnds[i][1]*x.sum()) # Technical constraint (max)
    model.st(k[:,i]@(x.sum(axis=1)) >= bnds[i][0]*x.sum()) # Technical constraint (min)
model.st(Vr*N <= bnds[7][1]*x.sum()) # HRT constraint (max)
model.st(Vr*N >= bnds[7][0]*x.sum()) # HRT constraint (min)
model.st(x >= 0)

# Add constraints from regulating autorithy
model.st(x[table1A].sum() >= 0.7*x.sum())
model.st(x[table1B].sum() <= 0.3*x.sum())
model.st(x[secondharvest].sum() <= 0.2*x.sum())
model.st(x[slurries].sum() >= 0.4*x.sum())

#Add constraint to consume substrate of 'myplant'
model.st(x.T[-1] == A[-1])

#Solve the model
model.solve(my_solver)
opt_obj = model.get()  #
opt_x = x.get()

print('The net revenue for the purchase horizon of interest is', round(opt_obj,2),
      'euro/day and the optimal fluxes of items to be purchased are', np.round(opt_x, decimals=2))

In [ ]:
# Compute quantities of interest with the obtained solution
x = opt_x
J = opt_obj
HRT = Vr*N/x.sum()
print(f'HRT is {HRT}')
f_obj = p_ch4*(VS*BMPinf/1000)@x.sum(axis=1)/N
f_obj2 = (Cp.T*x).sum()/N
f_obj3 = (Ct.T*x).sum()/N
print(f_obj)
print(f_obj2)
print(f_obj3)
print(f_obj-f_obj2-f_obj3)


#diet
diet = []
for i,name in zip(range(len(x)),data['Item characteristics'].values[:,0]):
    itempercentage = x[i].sum()/x.sum()*100
    print(f'{name} is {round(itempercentage,2)}% of the influent diet')
    diet.append(round(itempercentage,2))

# Save to Excel
results = [HRT, f_obj, f_obj2, f_obj3, J, diet]
file_path = 'Results.xlsx'
sheet_name = 'D-LP'
save_list_to_excel(results, file_path, sheet_name)

# Check which constraint is active
table1A = np.array([s for f, s in zip(itemtype, x) if any(target in f for target in ['S','B'])]).sum()
table1B = np.array([s for f, s in zip(itemtype, x) if any(target in f for target in ['P'])]).sum()
slurries = np.array([s for f, s in zip(itemtype, x) if any(target in f for target in ['S'])]).sum()
secondharvest = np.array([s for f, s in zip(itemtype, x) if any(target in f for target in ['P2'])]).sum()
deltarl = [table1A*100/x.sum() - 70, 30 - table1B*100/x.sum(), slurries*100/x.sum() - 40, 20 - secondharvest*100/x.sum()]
for i in range(len(deltarl)):
    active = [deltarl[i] if deltarl[i]<0 else 0]
    print(f'Autorithy constraint {i} not respected by {round(active[0],2)}')

#
for i in range(np.size(k,1)):
    deltamax = k[:,i]@(x.sum(axis=1))/x.sum() - bnds[i][1] # Technical constraint (max)
    deltamin = k[:,i]@(x.sum(axis=1))/x.sum() - bnds[i][0] # Technical constraint (min)
    print(f'Technical constriant {i} has margin to UB equal to {round(deltamax,2)}')
    print(f'Technical constriant {i} has margin to LB equal to {round(deltamin,2)}')
deltaHRTmax = Vr*N/x.sum() - bnds[7][1] # HRT constraint (max)
print(f'HRT constriant has margin to UB equal to {round(deltaHRTmax,2)}')
deltaHRTmin = Vr*N/x.sum() - bnds[7][0] # HRT constraint (min)
print(f'HRT constriant has margin to LB equal to {round(deltaHRTmin,2)}')

"More realistic approach" (with BMP = BMP(HRT))

In [ ]:
#Solve the nominal problem + HRT optimization

# Find indices correspondent to a certain cluster to apply 'regulating authority constraints'
table1A = np.isin(itemtype,['S','B'])
table1B = np.isin(itemtype,['P'])
slurries = np.isin(itemtype,['S'])
secondharvest = np.isin(itemtype,['P2'])

#Extract value from 'bmp_curve' at the given HRT
HRT=20
HRTminus1 = 1/HRT
Nminus1 = 1/N
BMP = []
for i in range(len(data['Item characteristics'].values[:,0])):
    BMPsubstrate = bmp_curve[i][HRT-1]
    BMP.append(BMPsubstrate)
BMP = np.array(BMP)

#Create model
model=ro.Model('Biomethane supply-chain real')

#Define variables
x = model.dvar((n,m))          #

#List the objective and constraints
model.max(Nminus1*(p_ch4*(VS*BMP/1000)@x.sum(axis=1)-(Cp.T*x).sum()-(Ct.T*x).sum())) #How shall I reformulate the objective function?
#model.st(x <= A)       # Stock constraint
model.st(x <= A.T)       # Availability constraint
for i in range(np.size(k,1)):
    model.st(k[:,i]@(x.sum(axis=1)) <= bnds[i][1]*x.sum()) # Technical constraint (max)
    model.st(k[:,i]@(x.sum(axis=1)) >= bnds[i][0]*x.sum()) # Technical constraint (min)
model.st(Vr*N == HRT*x.sum()) # HRT hard constraint
model.st(x >= 0)

#Add constraint to consume substrate of 'myplant'
model.st(x.T[-1] == A[-1])

#Add constraint to limit output residual BMP in digestate (authority limit)
model.st((BMPinf-BMP)@(x.sum(axis=1)) <= (0.15*BMPinf)@(x.sum(axis=1)))

# Add constraints from regulating autorithy
model.st(x[table1A].sum() >= 0.7*x.sum())
model.st(x[table1B].sum() <= 0.3*x.sum())
model.st(x[secondharvest].sum() <= 0.2*x.sum())
model.st(x[slurries].sum() >= 0.4*x.sum())

#Solve the model
model.solve(my_solver)
opt_obj_real = model.get()  #
opt_x_real = x.get()

print('The net revenue for the purchase horizon of interest is', round(opt_obj_real,2),
      'euro/day and the optimal fluxes of items to be purchased are', np.round(opt_x_real, decimals=2))

In [ ]:
# Compute quantities of interest with the obtained solution
x = opt_x_real
J = opt_obj_real
HRT = Vr*N/x.sum()
print(f'HRT is {HRT}')
f_obj = p_ch4*(VS*BMP/1000)@x.sum(axis=1)/N
f_obj2 = (Cp.T*x).sum()/N
f_obj3 = (Ct.T*x).sum()/N
print(f_obj)
print(f_obj2)
print(f_obj3)
print(f_obj-f_obj2-f_obj3)


#diet
diet = []
for i,name in zip(range(len(x)),data['Item characteristics'].values[:,0]):
    itempercentage = x[i].sum()/x.sum()*100
    print(f'{name} is {round(itempercentage,2)}% of the influent diet')
    diet.append(round(itempercentage,2))

# Save to Excel
results = [HRT, f_obj, f_obj2, f_obj3, J, diet]
file_path = 'Results.xlsx'
sheet_name = 'D-LP_real'
save_list_to_excel(results, file_path, sheet_name)

#
table1A = np.array([s for f, s in zip(itemtype, x) if any(target in f for target in ['S','B'])]).sum()
table1B = np.array([s for f, s in zip(itemtype, x) if any(target in f for target in ['P'])]).sum()
slurries = np.array([s for f, s in zip(itemtype, x) if any(target in f for target in ['S'])]).sum()
secondharvest = np.array([s for f, s in zip(itemtype, x) if any(target in f for target in ['P2'])]).sum()
deltarl = [table1A*100/x.sum() - 70, 30 - table1B*100/x.sum(), slurries*100/x.sum() - 40, 20 - secondharvest*100/x.sum()]
for i in range(len(deltarl)):
    active = [deltarl[i] if deltarl[i]<0 else 0]
    print(f'Autorithy constraint {i} not respected by {round(active[0],2)}')

#
for i in range(np.size(k,1)):
    deltamax = k[:,i]@(x.sum(axis=1))/x.sum() - bnds[i][1] # Technical constraint (max)
    deltamin = k[:,i]@(x.sum(axis=1))/x.sum() - bnds[i][0] # Technical constraint (min)
    print(f'Technical constriant {i} has margin to UB equal to {round(deltamax,2)}')
    print(f'Technical constriant {i} has margin to LB equal to {round(deltamin,2)}')
deltaHRTmax = Vr*N/x.sum() - bnds[7][1] # HRT constraint (max)
print(f'HRT constriant has margin to UB equal to {round(deltaHRTmax,2)}')
deltaHRTmin = Vr*N/x.sum() - bnds[7][0] # HRT constraint (min)
print(f'HRT constriant has margin to LB equal to {round(deltaHRTmin,2)}')

### Design of uncertainty sets

In [ ]:
# Design of uncertainty set
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Load mean, maximum and minimum
# Of prices
price_data = data['Purchase prices']
barley = price_data['Barley silage'].values.astype('float64')[0:3]
print(barley)
chicken = price_data['Chicken dung'].values.astype('float64')[0:3]
print(chicken)
cow_d = price_data['Cow dung'].values.astype('float64')[0:3]
print(cow_d)
cow_s = price_data['Cow slurry'].values.astype('float64')[0:3]
print(cow_s)
maize= price_data['Maize silage'].values.astype('float64')[0:3]
print(maize)
pig = price_data['Pig slurry'].values.astype('float64')[0:3]
print(pig)
sorghum = price_data['Sorghum silage'].values.astype('float64')[0:3]
print(sorghum)
tomato = price_data['Tomato peels'].values.astype('float64')[0:3]
print(tomato)
triticale= price_data['Triticale silage'].values.astype('float64')[0:3]
print(triticale)
wheat = price_data['Wheat byproducts'].values.astype('float64')[0:3]
print(wheat)

# Of availabilities
availability_data = data['Availability']
barley_a = availability_data['Barley silage'].values.astype('float64')[0:3]
print(barley_a)
chicken_a = availability_data['Chicken dung'].values.astype('float64')[0:3]
print(chicken_a)
cow_d_a = availability_data['Cow dung'].values.astype('float64')[0:3]
print(cow_d_a)
cow_s_a = availability_data['Cow slurry'].values.astype('float64')[0:3]
print(cow_s_a)
maize_a= availability_data['Maize silage'].values.astype('float64')[0:3]
print(maize_a)
pig_a = availability_data['Pig slurry'].values.astype('float64')[0:3]
print(pig_a)
sorghum_a = availability_data['Sorghum silage'].values.astype('float64')[0:3]
print(sorghum_a)
tomato_a = availability_data['Tomato peels'].values.astype('float64')[0:3]
print(tomato_a)
triticale_a = availability_data['Triticale silage'].values.astype('float64')[0:3]
print(triticale_a)
wheat_a = availability_data['Wheat byproducts'].values.astype('float64')[0:3]
print(wheat_a)
products_a = [barley_a,chicken_a,cow_d_a,cow_s_a,maize_a,pig_a,sorghum_a,tomato_a,triticale_a,wheat_a]

# Of BMP
bmp_data = [BMPinf,data['Item characteristics']['BMPinf Max'].values.astype('float64'),data['Item characteristics']['BMPinf Min'].values.astype('float64')]
print(bmp_data)

In [ ]:
# Function to peak a random realization of price
def samples(vector):
  mean = vector[0]
  max = vector[1]
  min = vector[2]

  alpha = max-min
  beta = alpha*(max-min)/(mean-min)-alpha
  return (max-min)*np.random.beta(alpha,beta)+min

# Function to generate prices data for one year in an incremental way
def one_year(product,plot=False):
  min = product[2]
  max= product[1]
  price_year = np.zeros(52)
  price_year[0] = samples(product)
  for i in range(price_year.shape[0]-1):
    incremented= price_year[i]+np.random.triangular(-0.5,0,0.5)
    if incremented <=min:
      price_year[i+1] = min
    elif incremented >=max:
      price_year[i+1] = max
    else:
      price_year[i+1] = incremented

  if plot:
    plt.plot(np.arange(1,len(price_year)+1),price_year)
  return price_year

In [ ]:
# Generation of data for prices
# 5 years with 52 realization/year (one/week)
np.random.seed(5)
product_prices_over_time=np.zeros([10,52*5])
for j in [0,52,52*2,52*3,52*4]:
  d=one_year(barley,False)
  product_prices_over_time[0,j:j+52]=d
plt.plot(np.arange(1,len(product_prices_over_time[0])+1),product_prices_over_time[0],label='barley silage')
for j in [0,52,52*2,52*3,52*4]:
  d=one_year(chicken,False)
  product_prices_over_time[1,j:j+52]=d
plt.plot(np.arange(1,len(product_prices_over_time[1])+1),product_prices_over_time[1],label='chicken dung')
for j in [0,52,52*2,52*3,52*4]:
  d=one_year(cow_d,False)
  product_prices_over_time[2,j:j+52]=d
plt.plot(np.arange(1,len(product_prices_over_time[2])+1),product_prices_over_time[2],label='cow dung')
for j in [0,52,52*2,52*3,52*4]:
  d=one_year(cow_s,False)
  product_prices_over_time[3,j:j+52]=d
plt.plot(np.arange(1,len(product_prices_over_time[3])+1),product_prices_over_time[3],label='cow slurry')
for j in [0,52,52*2,52*3,52*4]:
  d=one_year(maize,False)
  product_prices_over_time[4,j:j+52]=d
plt.plot(np.arange(1,len(product_prices_over_time[4])+1),product_prices_over_time[4],label='maize silage')
for j in [0,52,52*2,52*3,52*4]:
  d=one_year(pig,False)
  product_prices_over_time[5,j:j+52]=d
plt.plot(np.arange(1,len(product_prices_over_time[5])+1),product_prices_over_time[5],label='pig slurry')
for j in [0,52,52*2,52*3,52*4]:
  d=one_year(sorghum,False)
  product_prices_over_time[6,j:j+52]=d
plt.plot(np.arange(1,len(product_prices_over_time[6])+1),product_prices_over_time[6],label='sorghum silage')
for j in [0,52,52*2,52*3,52*4]:
  d=one_year(tomato,False)
  product_prices_over_time[7,j:j+52]=d
plt.plot(np.arange(1,len(product_prices_over_time[7])+1),product_prices_over_time[7],label='tomato peels')
for j in [0,52,52*2,52*3,52*4]:
  d=one_year(triticale,False)
  product_prices_over_time[8,j:j+52]=d
plt.plot(np.arange(1,len(product_prices_over_time[8])+1),product_prices_over_time[8],label='triticale silage')
for j in [0,52,52*2,52*3,52*4]:
  d=one_year(wheat,False)
  product_prices_over_time[9,j:j+52]=d
#plt.plot(np.arange(1,len(product_prices_over_time[9])+1),product_prices_over_time[9],label='wheat')
#filename = "wheat_price.csv"
#np.savetxt(filename, product_prices_over_time, delimiter=",")
plt.legend(bbox_to_anchor=(1, 1), loc='upper left')
plt.xlabel('Time (week)')
plt.ylabel('Purchase cost (€ ton$_{FM}^{-1}$)')
plt.grid()
plt.xlim(0,260)

plt.savefig(f'Pricedata_{current_date}.png', bbox_inches='tight')

# Data generation of availability
# One realization for each year
product_availability_over_time=np.zeros([10,5])#
row=0
for i in products_a:
  for j in range(5):
    product_availability_over_time[row,j] = samples(i)
  row+=1
product_availability_over_time

In [ ]:
# Design of uncertainty sets
# Budgeted sets

# Probability to meet chance constraint
Prob = 0.95
eps = 1-Prob

#Define Gamma_price
mu_price=np.mean(product_prices_over_time, axis=1)
tmp = product_prices_over_time - mu_price.reshape(-1,1)@np.ones((1,5*52))
tmp_max = abs(tmp).max(1)
Zs_price = np.diag(1/tmp_max)@tmp
P_price = np.diag(tmp_max)
tmp=np.sort(np.linalg.norm(Zs_price,1,axis=0))
Gamma_price=tmp[int(np.ceil((1-eps)*len(tmp)))-1]
print('Calibrating the budgeted set: Gamma_price={0:0.6f}'.format(Gamma_price))

#Defne Gamma_Availability
mu_avail=np.mean(product_availability_over_time, axis=1)
tmp = product_availability_over_time - mu_avail.reshape(-1,1)@np.ones((1,5))
tmp_max = abs(tmp).max(1)
Zs_avail = np.diag(1/tmp_max)@tmp
P_avail = np.diag(tmp_max)
tmp=np.sort(np.linalg.norm(Zs_avail,1,axis=0))
Gamma_availability=tmp[int(np.ceil((1-eps)*len(tmp)))-1]
print('Calibrating the budgeted set: Gamma_availability={0:0.6f}'.format(Gamma_availability))

#Define Gamma_BMP
# Calibrating the budgeted set based on theoretical result for symmetric [-1,1] distributions
Gamma_BMP = (2*n*np.log(1/eps))**0.5
print('Calibrating the budgeted set: Gamma_BMP={0:0.6f}'.format(Gamma_BMP))


### Robust problem

In [ ]:
# Check constraint formulation for availability peaking one realization from Zs
deterministic_mean_avail = [arr[0] for arr in products_a]
sellersizes = []
for i in range(n):
    column_a = A[:,i]/deterministic_mean_avail[i]
    sellersizes.append(column_a)
    #print(column_a.sum())
sellersizes = np.stack(sellersizes)
#print(sellersize[0])
#print(A[:,0])
Z = [arr[0] for arr in Zs_avail]
#print((mu_avail+P_avail@Z))
Anew = sellersizes.T*(mu_avail+P_avail@Z)
#print(Anew)

# Check constraint formulation for prices peaking one realization from Zs
item_presence = np.where(A != 0, 1, 0)
Z = [arr[0] for arr in Zs_price]
Cpnew = item_presence*(mu_price+P_price@Z)
#print(Cpnew[0])
#print(Cp[0])

# Compute maximum variation of BMP (toward direction of interest for robustification)
bmpdev = 1 - bmp_data[2]/bmp_data[0]
#print(bmpdev)

"First simplified approach" (BMP = BMP$_{inf}$, no HRT effect)

In [ ]:
#Solve the robustified problem

# Find indices correspondent to a certain cluster to apply 'regulating authority constraints'
table1A = np.isin(itemtype,['S','B'])
table1B = np.isin(itemtype,['P'])
slurries = np.isin(itemtype,['S'])
secondharvest = np.isin(itemtype,['P2'])

# Which uncertainty is considered?
uncertain_bmp = False
uncertain_price = True

#Create model
model=ro.Model('Biomethane supply-chain robust simplified')

#Define variables
s = model.dvar(1) #Lower-bound for revenues
c = model.dvar(1) #Ubber-bound for costs
x = model.dvar((n,m)) #Decision variable
z = model.rvar(n) #Random variables = perturbation to availabilities
w = model.rvar(n) #Random variables = perturbation to prices
l = model.rvar(n) #Random variables = perturbation to BMP

#Define 'availability uncertainty set'
availabilitySet = (-1<=z, z<=1, rso.norm(z,1)<=Gamma_availability)
#Define 'prices uncertainty set'
pricesSet = (-1<=w, w<=1, rso.norm(w,1)<=Gamma_price)
#Define 'BMP uncertainty set'
BMPSet = (-1<=l, l<=1, rso.norm(l,1)<=Gamma_BMP)

#List the objective and constraints
model.max(s-c)
if uncertain_bmp == True:
  model.st((s <= Nminus1*(p_ch4*(VS*BMPinf*(1+bmpdev*l)*(1/1000))@x.sum(axis=1))).forall(BMPSet)) #uncomment to include BMP uncertainty
else:
  model.st(s <= Nminus1*(p_ch4*(VS*BMPinf*(1/1000))@x.sum(axis=1))) #comment to include BMP uncertainty
if uncertain_price == True:
  model.st((c >= Nminus1*(((item_presence*(mu_price+P_price@w)).T*x).sum() + (Ct.T*x).sum())).forall(pricesSet)) #uncomment to include price uncertainty
else:
  model.st(c >= Nminus1*((Cp.T*x).sum() + (Ct.T*x).sum())) #comment to include price uncertainty

model.st((x[:,0:-1] <= (sellersizes[:,0:-1].T*(mu_avail+P_avail@z)).T).forall(availabilitySet))       # Availability constraint

for i in range(np.size(k,1)):
    model.st(k[:,i]@(x.sum(axis=1)) <= bnds[i][1]*x.sum()) # Technical constraint (max)
    model.st(k[:,i]@(x.sum(axis=1)) >= bnds[i][0]*x.sum()) # Technical constraint (min)
model.st(Vr*N <= bnds[7][1]*x.sum()) # HRT constraint (max)
model.st(Vr*N >= bnds[7][0]*x.sum()) # HRT constraint (min)
model.st(x >= 0)

#Add constraint to consume substrate of 'myplant'
model.st((x.T[-1] == A[-1]))

# Add constraints from regulating autorithy
model.st(x[table1A].sum() >= 0.7*x.sum())
model.st(x[table1B].sum() <= 0.3*x.sum())
model.st(x[secondharvest].sum() <= 0.2*x.sum())
model.st(x[slurries].sum() >= 0.4*x.sum())

#Solve the model
model.solve(my_solver)
opt_obj_robust = model.get()  #
opt_x_robust = x.get()
opt_s = s.get()
opt_c = c.get()

print('The net revenue for the purchase horizon of interest is', round(opt_obj_robust,2),
      'euro/day and the optimal fluxes of items to be purchased are', np.round(opt_x_robust, decimals=2))

In [ ]:
# Compute quantities of interest with the obtained solution
x = opt_x_robust
s = opt_s
c = opt_c
J = opt_obj_robust
HRT = Vr*N/x.sum()
print(f'HRT is {HRT}')
f_obj = s[0]
f_obj3 = (Ct.T*x).sum()/N
f_obj2 = c[0] - f_obj3
print(f_obj)
print(f_obj2)
print(f_obj3)
print(f_obj-f_obj2-f_obj3)


#diet
diet = []
for i,name in zip(range(len(x)),data['Item characteristics'].values[:,0]):
    itempercentage = x[i].sum()/x.sum()*100
    print(f'{name} is {round(itempercentage,2)}% of the influent diet')
    diet.append(round(itempercentage,2))

# Save to Excel
results = [Gamma_availability, Gamma_price, Gamma_BMP, HRT, f_obj, f_obj2, f_obj3, J, diet, uncertain_bmp, uncertain_price]
file_path = 'Results.xlsx'
sheet_name = 'R-LP'
save_list_to_excel(results, file_path, sheet_name)

#
table1A = np.array([s for f, s in zip(itemtype, x) if any(target in f for target in ['S','B'])]).sum()
table1B = np.array([s for f, s in zip(itemtype, x) if any(target in f for target in ['P'])]).sum()
slurries = np.array([s for f, s in zip(itemtype, x) if any(target in f for target in ['S'])]).sum()
secondharvest = np.array([s for f, s in zip(itemtype, x) if any(target in f for target in ['P2'])]).sum()
deltarl = [table1A*100/x.sum() - 70, 30 - table1B*100/x.sum(), slurries*100/x.sum() - 40, 20 - secondharvest*100/x.sum()]
for i in range(len(deltarl)):
    active = [deltarl[i] if deltarl[i]<0 else 0]
    print(f'Autorithy constraint {i} not respected by {round(active[0],2)}')

#
for i in range(np.size(k,1)):
    deltamax = k[:,i]@(x.sum(axis=1))/x.sum() - bnds[i][1] # Technical constraint (max)
    deltamin = k[:,i]@(x.sum(axis=1))/x.sum() - bnds[i][0] # Technical constraint (min)
    print(f'Technical constriant {i} has margin to UB equal to {round(deltamax,2)}')
    print(f'Technical constriant {i} has margin to LB equal to {round(deltamin,2)}')
deltaHRTmax = Vr*N/x.sum() - bnds[7][1] # HRT constraint (max)
print(f'HRT constriant has margin to UB equal to {round(deltaHRTmax,2)}')
deltaHRTmin = Vr*N/x.sum() - bnds[7][0] # HRT constraint (min)
print(f'HRT constriant has margin to LB equal to {round(deltaHRTmin,2)}')

"More realistic approach" (with BMP = BMP(HRT))

In [ ]:
#Solve the robustified problem

# Find indices correspondent to a certain cluster to apply 'regulating authority constraints'
table1A = np.isin(itemtype,['S','B'])
table1B = np.isin(itemtype,['P'])
slurries = np.isin(itemtype,['S'])
secondharvest = np.isin(itemtype,['P2'])

#Extract value from 'bmp_curve' at the given HRT
HRT=20
HRTminus1 = 1/HRT
BMP = []
for i in range(len(data['Item characteristics'].values[:,0])):
    BMPsubstrate = bmp_curve[i][HRT-1]
    BMP.append(BMPsubstrate)

# Which uncertainty is considered?
uncertain_bmp = True
uncertain_price = True

# Which Gamma is considered
#deltaGamma = 0.1
#Gamma_availability = Gamma_availability*(1+deltaGamma)
#Gamma_price = 0.0
#Gamma_BMP = Gamma_BMP*(1+deltaGamma)

#Create model
model=ro.Model('Biomethane supply-chain robust real')

#Define variables
s = model.dvar(1) #Lower-bound for revenues
c = model.dvar(1) #Ubber-bound for costs
x = model.dvar((n,m)) #Decision variable
z = model.rvar(n) #Random variables = perturbation to availabilities
w = model.rvar(n) #Random variables = perturbation to prices
l = model.rvar(n) #Random variables = perturbation to BMP

#Define 'availability uncertainty set'
availabilitySet = (-1<=z, z<=1, rso.norm(z,1)<=Gamma_availability)
#Define 'prices uncertainty set'
pricesSet = (-1<=w, w<=1, rso.norm(w,1)<=Gamma_price)
#Define 'BMP uncertainty set'
BMPSet = (-1<=l, l<=1, rso.norm(l,1)<=Gamma_BMP)

#List the objective and constraints
model.max(s-c)
if uncertain_bmp == True:
  model.st((s <= Nminus1*(p_ch4*(VS*BMP*(1+bmpdev*l)*(1/1000))@x.sum(axis=1))).forall(BMPSet)) #uncomment to include BMP uncertainty
else:
  model.st(s <= Nminus1*(p_ch4*(VS*BMP*(1/1000))@x.sum(axis=1))) #comment to include BMP uncertainty
if uncertain_price == True:
  model.st((c >= Nminus1*(((item_presence*(mu_price+P_price@w)).T*x).sum() + (Ct.T*x).sum())).forall(pricesSet)) #uncomment to include price uncertainty
else:
  model.st(c >= Nminus1*((Cp.T*x).sum() + (Ct.T*x).sum())) #comment to include price uncertainty

model.st((x[:,0:-1] <= (sellersizes[:,0:-1].T*(mu_avail+P_avail@z)).T).forall(availabilitySet))       # Availability constraint

for i in range(np.size(k,1)):
    model.st(k[:,i]@(x.sum(axis=1)) <= bnds[i][1]*x.sum()) # Technical constraint (max)
    model.st(k[:,i]@(x.sum(axis=1)) >= bnds[i][0]*x.sum()) # Technical constraint (min)
model.st(Vr*N == HRT*x.sum()) # HRT hard constraint
model.st(x >= 0)

#Add constraint to consume substrate of 'myplant'
model.st((x.T[-1] == A[-1]))

#Add constraint to limit output residual BMP in digestate (authority limit)
model.st((((BMPinf-BMP)*(1+bmpdev*l))@(x.sum(axis=1)) <= (0.15*BMPinf*(1+bmpdev*l))@(x.sum(axis=1))).forall(BMPSet))

# Add constraints from regulating autorithy
model.st(x[table1A].sum() >= 0.7*x.sum())
model.st(x[table1B].sum() <= 0.3*x.sum())
model.st(x[secondharvest].sum() <= 0.2*x.sum())
model.st(x[slurries].sum() >= 0.4*x.sum())

#Solve the model
model.solve(my_solver)
opt_obj_robust_real = model.get()  #
opt_x_robust_real = x.get()
opt_s_real = s.get()
opt_c_real = c.get()

print('The net revenue for the purchase horizon of interest is', round(opt_obj_robust_real,2),
      'euro/day and the optimal fluxes of items to be purchased are', np.round(opt_x_robust_real, decimals=2))

In [ ]:
# Compute quantities of interest with the obtained solution
x = opt_x_robust_real
s = opt_s_real
c = opt_c_real
J = opt_obj_robust_real
HRT = Vr*N/x.sum()
print(f'HRT is {HRT}')
f_obj = s[0]
f_obj3 = (Ct.T*x).sum()/N
f_obj2 = c[0] - f_obj3
print(f_obj)
print(f_obj2)
print(f_obj3)
print(f_obj-f_obj2-f_obj3)


#diet
diet = []
for i,name in zip(range(len(x)),data['Item characteristics'].values[:,0]):
    itempercentage = x[i].sum()/x.sum()*100
    print(f'{name} is {round(itempercentage,2)}% of the influent diet')
    diet.append(round(itempercentage,2))

# Save to Excel
results = [Gamma_availability, Gamma_price, Gamma_BMP, HRT, f_obj, f_obj2, f_obj3, J, diet, uncertain_bmp, uncertain_price]
file_path = 'Results.xlsx'
sheet_name = 'R-LP_real'
save_list_to_excel(results, file_path, sheet_name)

#
table1A = np.array([s for f, s in zip(itemtype, x) if any(target in f for target in ['S','B'])]).sum()
table1B = np.array([s for f, s in zip(itemtype, x) if any(target in f for target in ['P'])]).sum()
slurries = np.array([s for f, s in zip(itemtype, x) if any(target in f for target in ['S'])]).sum()
secondharvest = np.array([s for f, s in zip(itemtype, x) if any(target in f for target in ['P2'])]).sum()
deltarl = [table1A*100/x.sum() - 70, 30 - table1B*100/x.sum(), slurries*100/x.sum() - 40, 20 - secondharvest*100/x.sum()]
for i in range(len(deltarl)):
    active = [deltarl[i] if deltarl[i]<0 else 0]
    print(f'Autorithy constraint {i} not respected by {round(active[0],2)}')

#
for i in range(np.size(k,1)):
    deltamax = k[:,i]@(x.sum(axis=1))/x.sum() - bnds[i][1] # Technical constraint (max)
    deltamin = k[:,i]@(x.sum(axis=1))/x.sum() - bnds[i][0] # Technical constraint (min)
    print(f'Technical constriant {i} has margin to UB equal to {round(deltamax,2)}')
    print(f'Technical constriant {i} has margin to LB equal to {round(deltamin,2)}')
deltaHRTmax = Vr*N/x.sum() - bnds[7][1] # HRT constraint (max)
print(f'HRT constriant has margin to UB equal to {round(deltaHRTmax,2)}')
deltaHRTmin = Vr*N/x.sum() - bnds[7][0] # HRT constraint (min)
print(f'HRT constriant has margin to LB equal to {round(deltaHRTmin,2)}')

### Testing

In [ ]:
# Generation of data for prices
# 1 years with 52 realization/year (one/week)
np.random.seed(5)
product_prices_test= []
d=np.array([samples(barley) for _ in range(20)])
product_prices_test.append(d)
d=np.array([samples(chicken) for _ in range(20)])
product_prices_test.append(d)
d=np.array([samples(cow_d) for _ in range(20)])
product_prices_test.append(d)
d=np.array([samples(cow_s) for _ in range(20)])
product_prices_test.append(d)
d=np.array([samples(maize) for _ in range(20)])
product_prices_test.append(d)
d=np.array([samples(pig) for _ in range(20)])
product_prices_test.append(d)
d=np.array([samples(sorghum) for _ in range(20)])
product_prices_test.append(d)
d=np.array([samples(tomato) for _ in range(20)])
product_prices_test.append(d)
d=np.array([samples(triticale) for _ in range(20)])
product_prices_test.append(d)
d=np.array([samples(wheat) for _ in range(20)])
product_prices_test.append(d)

In [ ]:
#At a given HRT...

def testJ(x,Cp):
    J = Nminus1*(p_ch4*(VS*BMP/1000)@x.sum(axis=1)-(Cp.T*x).sum()-(Ct.T*x).sum())
    return J

Cp_test_set = [item_presence*np.array(product_prices_test)[:,i] for i in range(20)]
J_test_robust_real = [testJ(opt_x_robust_real,Cp_test_set[i]) for i in range(20)]
J_test_deterministic_real = [testJ(opt_x_real,Cp_test_set[i]) for i in range(20)]

#----------------------------------------------------------------------------------------

# Plot the boxplot
plt.boxplot(J_test_robust_real, vert=True, positions=[1], meanline=True, showmeans=True, meanprops={'color': 'r', 'linewidth': 2})
plt.scatter([1], opt_obj_robust_real, color='b', label='R-LP J value')  # Highlight other value

plt.boxplot(J_test_deterministic_real, positions=[2], vert=True, meanline=True, showmeans=True, meanprops={'color': 'r', 'linewidth': 2})
plt.scatter([2], opt_obj_real, color='green', label='D-LP J value')  # Highlight other value

# Set labels and legend
plt.xticks([1, 2], ['R-LP', 'D-LP'])
plt.ylabel('J value (€ day$^{-1}$)')
plt.legend()

plt.savefig(f'Testing_{HRT}_{current_date}.png')

plt.show()

### Plot

In [ ]:
# Plot results
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Step 1: Read the Excel file and select the desired sheet
dataforplot = pd.read_excel('Results.xlsx', sheet_name='R-LP_real (2)')

# Filter rows
filtered_df = dataforplot.iloc[6:,:]

# Step 2: Select specific columns and rows
x_values = filtered_df['HRT']  # Select values from the first row, excluding the first column
y_values = filtered_df['J'] # Select values from the second row, excluding the first column

bar_columns = filtered_df.columns[8:18]
other_data = filtered_df[filtered_df.columns[8:18]]

# Create subplots with a single row and one column
fig, ax1 = plt.subplots()

# Plot the stacked bar chart
other_data.plot(kind='bar', stacked=True, ax=ax1)
ax1.set_ylim(0,100)
ax1.set_xlabel('HRT (days)')

ax1.set_xticklabels(round(x_values,2))

# Get the position and size of the first subplot
pos1 = ax1.get_position()

# Define the position and size of the new axes object
pos2 = [pos1.x0 + 1.3, pos1.y0, pos1.width, pos1.height]

# Create a new axes object within the same figure
ax2 = fig.add_axes(pos2)

# Plot the line chart
ax2.plot(x_values, y_values, marker='o', color='black')
ax2.set_xlabel('HRT (days)')
ax2.set_ylabel('Net profit (€ day$^{-1}$)')

# Adjust layout to prevent overlap
plt.tight_layout()

# Move the legend outside the left subplot
ax1.legend(bbox_to_anchor=(1, 1), loc='upper left')

plt.savefig(f'Screening_HRT_R-LP_{current_date}.png', bbox_inches='tight')

# Show plots
plt.show()

In [ ]:
# Step 1: Read the Excel file and select the desired sheet
dataforplot = pd.read_excel('Results.xlsx', sheet_name='D-LP_real (2)')

# Filter rows
filtered_df = dataforplot.iloc[:,:]

# Step 2: Select specific columns and rows
x_values = filtered_df['HRT']  # Select values from the first row, excluding the first column
y_values = filtered_df['J'] # Select values from the second row, excluding the first column

# Sort the data based on x_values
sorted_indices = sorted(range(len(x_values)), key=lambda k: x_values.values[k])
x_values_sorted = [x_values.values[i] for i in sorted_indices]
y_values_sorted = [y_values.values[i] for i in sorted_indices]

bar_columns = filtered_df.columns[8:18]
other_data = filtered_df[filtered_df.columns[8:18]]

# Create subplots with a single row and one column
fig, ax1 = plt.subplots()

# Plot the stacked bar chart
other_data= other_data.iloc[sorted_indices]
other_data.plot(kind='bar', stacked=True, ax=ax1)
ax1.set_ylim(0,100)
ax1.set_xlabel('HRT (days)')

rounded_values = list(map(lambda x: round(x, 2), x_values_sorted))
ax1.set_xticklabels(rounded_values)

# Get the position and size of the first subplot
pos1 = ax1.get_position()

# Define the position and size of the new axes object
pos2 = [pos1.x0 + 1.3, pos1.y0, pos1.width, pos1.height]

# Create a new axes object within the same figure
ax2 = fig.add_axes(pos2)

# Plot the line chart
ax2.plot(x_values_sorted, y_values_sorted, marker='o', color='black')
ax2.set_xlabel('HRT (days)')
ax2.set_ylabel('Net profit (€ day$^{-1}$)')

# Adjust layout to prevent overlap
plt.tight_layout()

# Move the legend outside the left subplot
ax1.legend(bbox_to_anchor=(1, 1), loc='upper left')

plt.savefig(f'Screening_HRT_D-LP_{current_date}.png', bbox_inches='tight')

# Show plots
plt.show()

In [ ]:
# Step 1: Read the Excel file and select the desired sheet
dataforplot = pd.read_excel('Results.xlsx', sheet_name='R-LP_real (3)')

# Filter rows
filtered_df = dataforplot.iloc[0:13,:] #!!!

# Step 2: Select specific columns and rows
x_values = filtered_df['VaR']  # Select values from the first row, excluding the first column
y_values = filtered_df['J'] # Select values from the second row, excluding the first column

# Sort the data based on x_values
sorted_indices = sorted(range(len(x_values)), key=lambda k: x_values.values[k])
x_values_sorted = [x_values.values[i] for i in sorted_indices]
y_values_sorted = [y_values.values[i] for i in sorted_indices]

bar_columns = filtered_df.columns[8:18]
other_data = filtered_df[filtered_df.columns[8:18]]

# Create subplots with a single row and one column
fig, ax1 = plt.subplots()

# Plot the stacked bar chart
other_data= other_data.iloc[sorted_indices]
other_data.plot(kind='bar', stacked=True, ax=ax1)
ax1.set_ylim(0,100)
ax1.set_xlabel('Prob (-)')

rounded_values = list(map(lambda x: round(x, 2), x_values_sorted))
ax1.set_xticklabels(rounded_values)

# Get the position and size of the first subplot
pos1 = ax1.get_position()

# Define the position and size of the new axes object
pos2 = [pos1.x0 + 1.3, pos1.y0, pos1.width, pos1.height]

# Create a new axes object within the same figure
ax2 = fig.add_axes(pos2)

# Plot the line chart
ax2.plot(x_values_sorted, y_values_sorted, marker='o', color='black')
ax2.set_xlabel('Prob (-)')
ax2.set_ylabel('Net profit (€ day$^{-1}$)')

# Adjust layout to prevent overlap
plt.tight_layout()

# Move the legend outside the left subplot
ax1.legend(bbox_to_anchor=(1, 1), loc='upper left')

plt.savefig(f'Screening_Prob_R-LP_20_{current_date}.png', bbox_inches='tight')

# Show plots
plt.show()

In [ ]:
# Step 1: Read the Excel file and select the desired sheet
dataforplot = pd.read_excel('Results.xlsx', sheet_name='R-LP_real (3)')

# Filter rows
filtered_df = dataforplot.iloc[13:,:]

# Step 2: Select specific columns and rows
x_values = filtered_df['VaR']  # Select values from the first row, excluding the first column
y_values = filtered_df['J'] # Select values from the second row, excluding the first column

# Sort the data based on x_values
sorted_indices = sorted(range(len(x_values)), key=lambda k: x_values.values[k])
x_values_sorted = [x_values.values[i] for i in sorted_indices]
y_values_sorted = [y_values.values[i] for i in sorted_indices]

bar_columns = filtered_df.columns[8:18]
other_data = filtered_df[filtered_df.columns[8:18]]

# Create subplots with a single row and one column
fig, ax1 = plt.subplots()

# Plot the stacked bar chart
other_data= other_data.iloc[sorted_indices]
other_data.plot(kind='bar', stacked=True, ax=ax1)
ax1.set_ylim(0,100)
ax1.set_xlabel('Prob (-)')

rounded_values = list(map(lambda x: round(x, 2), x_values_sorted))
ax1.set_xticklabels(rounded_values)

# Get the position and size of the first subplot
pos1 = ax1.get_position()

# Define the position and size of the new axes object
pos2 = [pos1.x0 + 1.3, pos1.y0, pos1.width, pos1.height]  # Move the new axes to the right by 0.5

# Create a new axes object within the same figure
ax2 = fig.add_axes(pos2)

# Plot the line chart
ax2.plot(x_values_sorted, y_values_sorted, marker='o', color='black')
ax2.set_xlabel('Prob (-)')
ax2.set_ylabel('Net profit (€ day$^{-1}$)')

# Adjust layout to prevent overlap
plt.tight_layout()

# Move the legend outside the left subplot
ax1.legend(bbox_to_anchor=(1, 1), loc='upper left')

plt.savefig(f'Screening_Prob_R-LP_50_{current_date}.png', bbox_inches='tight')

# Show plots
plt.show()

In [ ]:
fig, ax1 = plt.subplots(figsize=(8,6))
# Get the position and size of the first subplot
pos1 = ax1.get_position()

# Define the position and size of the new axes object
pos2 = [pos1.x0 + 0.9, pos1.y0, pos1.width, pos1.height]  # Move the new axes to the right by 0.5

# Create a new axes object within the same figure
ax2 = fig.add_axes(pos2)

for i in range(n-1):
  ax1.plot(1.138*VS[i]*bmp_curve[i]/1000-data['Purchase prices'].values[0,1:11].astype('float64')[i],label=data['Item characteristics'].values[:,0][i])
  ax1.set_xlabel('HRT (d)')
  ax1.set_xlim(0,50)
  ax1.set_ylabel('Net profit (€ ton$_{FM}^{-1}$)')
  ax1.grid()

  ax2.plot(bmp_curve[i],label=data['Item characteristics'].values[:,0][i])
  ax2.set_xlabel('HRT (d)')
  ax2.set_xlim(0,50)
  ax2.set_ylabel('BMP (Nm$^3_{ch4}$ ton$_{FM}^{-1}$)')
  ax2.legend(bbox_to_anchor=(1, 1), loc='upper left')
  ax2.grid()

plt.savefig(f'NetProfitBMP_{current_date}.png', bbox_inches='tight')

In [ ]:
# Real distribution of price data (standardized to Zs)
fig, ax = plt.subplots(1, 1)
for i in np.arange(10):
    ax.hist(Zs_price[i,:], density=True, bins='auto', histtype='stepfilled', alpha=0.2, label = data['Item characteristics'].values[:,0][i])
ax.legend(bbox_to_anchor=(1, 1), loc='upper left')
plt.savefig(f'Distribution_price_{current_date}.png', bbox_inches='tight')

In [ ]:
# Real distribution of price data (standardized to Zs)
# Distribution with 10 years realizations (Gamma_price(1%) was lower and Gamma_price(95%) was higher with respect to above-mentioned 5 years realizations)
fig, ax = plt.subplots(1, 1)
for i in np.arange(10):
    ax.hist(Zs_price[i,:], density=True, bins='auto', histtype='stepfilled', alpha=0.2, label = data['Item characteristics'].values[:,0][i])
ax.legend(bbox_to_anchor=(1, 1), loc='upper left')
#plt.savefig(f'Distribution_price_{current_date}.png', bbox_inches='tight')